# Lab 07-05 — EvaluationHarness A/B: BGE vs E5 Embeddings

**Track 07 · Evaluation** — the payoff: one harness, full RAGAS quartet, clean attribution.

The previous labs scored a pipeline with bespoke snippets. This lab runs the shared `EvaluationHarness` (`src/evaluation/harness.py`) — faithfulness, answer relevance, context precision, and context recall — over a hand-authored golden set, then does a controlled A/B: the **same corpus, questions, generator and judge**, with only the embedding model swapped (BGE vs E5).

```text
rag-mini first 800 passages
  -> retriever A: BGE  (default)
  -> retriever B: E5   (multilingual-e5-base)
  -> 6 golden questions, top_k = 3
  -> EvaluationHarness -> RAGAS quartet per variant
  -> aggregate means + verification gate (--verify)
```


## Setup

This notebook mirrors `src/curriculum/07-evaluation/05-harness-ab.py` exactly — the same verified code, split into cells. You can run it from anywhere: the imports cell walks up to the repo root and cd's into it, so every `Data/...` path resolves just like the lab script.

From the terminal, the lab runs as:

```bash
python src/curriculum/07-evaluation/05-harness-ab.py          # run + demo
python src/curriculum/07-evaluation/05-harness-ab.py --verify # verification gate
```

**LLM keys**: the lab judges with the local Ollama judge (no API key needed) and does not need a generator LLM. `GROQ_API_KEY` is not required for this lab.

The next cell installs the lab-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# is a no-op safety net for fresh environments):
#   sentence-transformers -> local BGE/E5 embeddings (embeddings/*.py)
#   faiss-cpu             -> the FAISS index (vectordb/faiss.py)
#   python-dotenv         -> repo-root .env handling
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu python-dotenv pandas


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory — this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) — then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))


import pandas as pd  # noqa: E402
from dotenv import load_dotenv  # noqa: E402
load_dotenv(REPO_ROOT / ".env")
from embeddings.bge import BGEEmbedding  # noqa: E402
from embeddings.e5 import E5Embedding  # noqa: E402
from evaluation.harness import EvaluationHarness  # noqa: E402
from evaluation.judge import LLMJudge  # noqa: E402
from evaluation.metrics import (  # noqa: E402
    AnswerRelevanceMetric,
    ContextPrecisionMetric,
    ContextRecallMetric,
    FaithfulnessMetric,
)
from langchain_core.documents import Document  # noqa: E402
from retrieval.similarity import SimilarityRetriever  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402



## 1. Configuration

The golden set is **6 hand-authored questions**, each validated at authoring time so the gold answer is contained verbatim in one of the first 800 passages (the corpus is capped to keep the A/B fast). `TOP_K = 3` context chunks per question; the two embedding models are named constants.


In [ ]:
# 1. Configuration
RAG_MINI = Path("Data/corpus/rag-mini-wikipedia")
PASSAGES_PATH = RAG_MINI / "passages.parquet"
N_PASSAGES = 800  # deterministic head of the corpus (keeps runtime low)
TOP_K = 3  # context chunks fed to the generator (harness.top_k)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
E5_MODEL_NAME = "intfloat/multilingual-e5-base"

# The golden set: hand-authored references, validated at authoring time so the
# gold answer appears verbatim in a retrievable passage of the corpus.
GOLDEN_QA: list[dict] = [
    {
        "doc": "rag-mini",
        "question": "Who assassinated Lincoln?",
        "reference": "John Wilkes Booth, Lincoln's assassin, can be seen in "
        "the crowd at Lincoln's second inauguration on March 4, 1865.",
    },
    {
        "doc": "rag-mini",
        "question": "The Celsius crater on the Moon is what?",
        "reference": "The Celsius crater on the Moon is named after the "
        "scientist Anders Celsius.",
    },
    {
        "doc": "rag-mini",
        "question": "What period of rapid economic growth did the United "
        "States experience during Coolidge's presidency?",
        "reference": "During Coolidge's presidency the United States "
        "experienced the period of rapid economic growth known as the "
        "Roaring Twenties.",
    },
    {
        "doc": "rag-mini",
        "question": "In 1905 Coolidge met and married whom?",
        "reference": "In 1905 Coolidge met and married Grace Anna Goodhue, "
        "a local schoolteacher and fellow Vermonter.",
    },
    {
        "doc": "rag-mini",
        "question": "When did Islam become the dominant religion in Java "
        "and Sumatra?",
        "reference": "Islam became the dominant religion in Java and "
        "Sumatra by the end of the 16th century.",
    },
    {
        "doc": "rag-mini",
        "question": "Who was on the committee with Adams to draft a "
        "Declaration of Independence?",
        "reference": "Adams was appointed on a committee with Thomas "
        "Jefferson, Benjamin Franklin, Robert R. Livingston and Roger "
        "Sherman to draft a Declaration of Independence.",
    },
]


## 2. Build one retriever per embedding model

`build_retriever(model)` embeds the 800 passages, builds a FAISS index, and returns a `SimilarityRetriever`. The embedder runs on CPU (`device="cpu"`) — the shared GPU is held by the local Ollama judge. Two calls, two retrievers; nothing else differs.


In [ ]:
# 2. Build one retriever per embedding model
def build_retriever(embedder, passages: list[str]) -> SimilarityRetriever:
    """Index the passages and return a top-k similarity retriever."""
    vectors = embedder.embed_documents(passages)
    chunks = [
        Document(page_content=t, metadata={"id": i})
        for i, t in enumerate(passages)
    ]
    store = FAISSVectorStore(embedding=embedder)
    store.add(chunks, embeddings=vectors)
    return SimilarityRetriever(store, top_k=TOP_K)


## 3. Experiment — run the harness once per variant

The `EvaluationHarness` runs each variant end-to-end and scores every (question, context, answer) triple on the full RAGAS quartet:

- **faithfulness** — claims in the answer supported by the context
- **answer_relevance** — does the answer address the question?
- **context_precision** — is each retrieved chunk relevant to the question?
- **context_recall** — did retrieval surface the passage that supports the gold answer?

Retrieval quality shows up in context precision/recall; generation quality in faithfulness and answer relevance.


In [ ]:
# 3. Experiment — run the harness once per variant
def run_experiment() -> dict:
    df = pd.read_parquet(PASSAGES_PATH)
    passages = [str(r["passage"]).strip() for _, r in df.iterrows()][
        :N_PASSAGES
    ]

    # device="cpu": the local Ollama judge holds the shared GPU (5.6 GiB
    # here); bulk-embedding on CPU avoids CUDA OOM and leaves the GPU for the
    # judge calls that follow.
    bge = BGEEmbedding(model_name=BGE_MODEL_NAME, device="cpu")
    e5 = E5Embedding(model_name=E5_MODEL_NAME, device="cpu")

    t0 = time.perf_counter()
    retriever_bge = build_retriever(bge, passages)
    retriever_e5 = build_retriever(e5, passages)
    embed_s = time.perf_counter() - t0

    judge = LLMJudge()
    harness_kwargs = dict(
        judge=judge,
        faithfulness_metric=FaithfulnessMetric(judge),
        answer_relevance_metric=AnswerRelevanceMetric(judge),
        context_precision_metric=ContextPrecisionMetric(judge),
        context_recall_metric=ContextRecallMetric(judge),
        top_k=TOP_K,
    )

    t0 = time.perf_counter()
    results_bge = EvaluationHarness(
        retriever=retriever_bge, **harness_kwargs
    ).run(GOLDEN_QA)
    results_e5 = EvaluationHarness(
        retriever=retriever_e5, **harness_kwargs
    ).run(GOLDEN_QA)
    eval_s = time.perf_counter() - t0

    return {
        "passages": len(passages),
        "embed_s": embed_s,
        "eval_s": eval_s,
        "results_bge": results_bge,
        "results_e5": results_e5,
    }


## 4. Demo

The demo prints both variants' per-question rows and the A/B means table. Reading the rows beats reading the means: one bad retrieval on a single question explains a dip better than an average ever will. Expect BGE to win context_recall — the E5 miss on one passage is visible as a 0 on that row.


In [ ]:
# 4. Demo
def print_demo(exp: dict) -> None:
    print("=" * 72)
    print("Lab 07-05 — EvaluationHarness A/B: BGE vs E5 embeddings")
    print(f"rag-mini {exp['passages']} passages, "
          f"{len(exp['results_bge'])} golden questions, top_k={TOP_K}")
    print("=" * 72)

    harness = EvaluationHarness(
        judge=None, retriever=None, faithfulness_metric=None,
        answer_relevance_metric=None, context_precision_metric=None,
        context_recall_metric=None, top_k=TOP_K,
    )
    agg_bge = harness.aggregate(exp["results_bge"])["overall"]
    agg_e5 = harness.aggregate(exp["results_e5"])["overall"]

    print(f"\n[1] Per-question rows — variant A (BGE):")
    harness.print_table(exp["results_bge"])
    print(f"\n[2] Per-question rows — variant B (E5):")
    harness.print_table(exp["results_e5"])

    print(f"\n[3] A/B means (RAGAS quartet):")
    print(f"    {'metric':<20} {'A (BGE)':>9} {'B (E5)':>9}  delta")
    for key in ("faithfulness", "answer_relevance",
                "context_precision", "context_recall"):
        diff = agg_bge[key] - agg_e5[key]
        print(f"    {key:<20} {agg_bge[key]:>9.3f} {agg_e5[key]:>9.3f}  "
              f"{diff:+.3f}")

    print(f"\n[4] Timing: embed {exp['embed_s']:.0f}s, harness both "
          f"{exp['eval_s']:.0f}s")

    print(f"\n[5] Takeaway")
    print("    A/B on embeddings: same corpus, same questions, same judge,")
    print("    same generator — only the embedder differs, so metric deltas")
    print("    are attributable. The harness gives the full RAGAS quartet")
    print("    instead of a single number: retrieval quality shows up in")
    print("    context precision/recall, generation quality in faithfulness")
    print("    and answer relevance. Reading the rows beats reading the")
    print("    means: one bad retrieval on a question explains a dip better")
    print("    than a mean ever will. Note the judge is the same Ollama")
    print("    model used for both variants — the comparison stays fair even")
    print("    though the judge is not interchangeable with a reference.")
    print("    This is the same harness you would wire into a golden-set")
    print("    regression (lab 04) for every embedding/retriever change.")


## 5. Verification gate

The gate checks both variants ran exactly the golden set, every metric lands in [0, 1], both retrievers actually answer the golden set (`context_recall > 0`), and the A/B is *meaningful* — the variants differ on at least one metric by more than 0.01.


In [ ]:
# 5. Verification gate
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    harness = EvaluationHarness(
        judge=None, retriever=None, faithfulness_metric=None,
        answer_relevance_metric=None, context_precision_metric=None,
        context_recall_metric=None, top_k=TOP_K,
    )
    agg_bge = harness.aggregate(exp["results_bge"])["overall"]
    agg_e5 = harness.aggregate(exp["results_e5"])["overall"]

    for label, results in (("BGE", exp["results_bge"]),
                           ("E5", exp["results_e5"])):
        checks.append(
            (f"{label}: {len(results)} questions evaluated (== 6)",
             len(results) == len(GOLDEN_QA)))
        for key in ("faithfulness", "answer_relevance",
                    "context_precision", "context_recall"):
            checks.append((f"{label} {key} in [0, 1]",
                           0.0 <= harness.aggregate(results)["overall"][key]
                           <= 1.0))

    # The golden set must be answerable by both variants (validated at
    # authoring time: gold contained in a retrievable passage).
    checks.append(("BGE context_recall > 0 (golden set is answerable)",
                   agg_bge["context_recall"] > 0.0))
    checks.append(("E5 context_recall > 0 (golden set is answerable)",
                   agg_e5["context_recall"] > 0.0))

    # The A/B is meaningful only if the variants differ somewhere.
    differs = any(abs(agg_bge[k] - agg_e5[k]) > 0.01
                  for k in ("faithfulness", "answer_relevance",
                            "context_precision", "context_recall"))
    checks.append(("A/B meaningful: variants differ on >= 1 metric", differs))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Embedding 800 passages twice on CPU takes under a minute; the harness's judge calls over 6 questions take a couple of minutes. `exp` holds both variants' result lists.


In [ ]:
exp = run_experiment()


### Demo — the artifact


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces.


In [ ]:
verify_gate(exp)
